# 2.4 开发环境部署

本节介绍 CANN Toolkit 和 Ops 包的安装部署，以及环境变量的配置。

---

## 学习目标

完成本节后，你将能够：
- 安装 CANN Toolkit 和 Ops 包到指定路径
- 配置环境变量并使其生效
- 验证安装结果

---

## 1. 安装包说明

CANN 开发环境由两个独立的 `.run` 包组成：

| 包 | 文件名格式 | 说明 |
|------|------|------|
| **Toolkit 包** | `Ascend-cann-toolkit_<version>_linux-<arch>.run` | 提供编译器（bisheng）、头文件、运行时库、工具链 |
| **Ops 包** | `Ascend-cann-<soc>-ops_<version>_linux-<arch>.run` | 提供预编译的算子二进制文件（运行态依赖） |

说明：
- `<version>`：CANN 版本号，如 9.0.0
- `<arch>`：CPU 架构，`aarch64` 或 `x86_64`
- `<soc>`：芯片型号，如 `950`

---

```mermaid
flowchart TB
    subgraph 准备["1. 准备阶段"]
        DL["下载 Toolkit .run 包"]
        DL2["下载 Ops .run 包"]
    end
    subgraph 安装["2. 安装阶段"]
        CHMOD["chmod +x *.run"]
        TK["安装 Toolkit<br/>--install --full<br/>--install-path=..."]
        OPS["安装 Ops 包<br/>--install<br/>--install-path=...<br/>路径必须与Toolkit一致"]
    end
    subgraph 配置["3. 配置阶段"]
        ENV["source set_env.sh<br/>设置环境变量"]
        RC["追加到 ~/.bashrc<br/>永久生效"]
    end
    subgraph 验证["4. 验证阶段"]
        VER1["bishengc++ --version"]
        VER2["npu-smi info"]
    end
    准备 --> 安装 --> 配置 --> 验证
```

## 2. 安装 Toolkit

### 2.1 默认路径安装

```bash
# 赋予可执行权限
chmod +x Ascend-cann-toolkit_9.0.0_linux-x86_64.run

# 安装（默认路径 /usr/local/Ascend）
./Ascend-cann-toolkit_9.0.0_linux-x86_64.run --install --full
```

### 2.2 指定路径安装

```bash
# 安装到自定义路径（如用户 HOME 目录）
./Ascend-cann-toolkit_9.0.0_linux-x86_64.run --install --full \
    --install-path=${HOME}/Ascend
```

- `--install`：执行安装
- `--full`：完整安装
- `--install-path`：指定安装目标路径

---

## 3. 安装 Ops 包

Ops 包提供算子的运行态依赖，安装路径必须与 Toolkit 一致。

```bash
chmod +x Ascend-cann-950-ops_9.0.0_linux-x86_64.run

# 与 Toolkit 安装到相同路径
./Ascend-cann-950-ops_9.0.0_linux-x86_64.run --install \
    --install-path=/usr/local/Ascend
```

安装完成后，目录结构如下：

```
/usr/local/Ascend/
├── ascend-toolkit/            # Toolkit 安装目录
│   └── latest/                # 指向当前版本的软链接
│       ├── bin/               # 编译器 bishengc++
│       ├── include/           # 头文件
│       ├── lib64/             # 运行时库
│       └── opp/               # 算子原型库（Ops 包安装到这里）
├── cann/
│   └── set_env.sh             # 环境变量脚本（统一入口）
└── driver/                    # 驱动
```

---

## 4. 环境变量配置

### 4.1 临时生效（当前终端）

```bash
# 默认路径
source /usr/local/Ascend/cann/set_env.sh

# 自定义路径
source ${HOME}/Ascend/cann/set_env.sh
```

`set_env.sh` 会自动设置 ASCEND_TOOLKIT_HOME、PATH、LD_LIBRARY_PATH、PYTHONPATH 等环境变量。

### 4.2 永久生效（每次登录自动加载）

```bash
# 追加到 ~/.bashrc
echo "source /usr/local/Ascend/cann/set_env.sh" >> ~/.bashrc
source ~/.bashrc
```

---

## 5. 验证安装

```bash
# 检查编译器版本
bishengc++ --version

# 检查 NPU 设备
npu-smi info

# 检查安装信息
cat /usr/local/Ascend/ascend-toolkit/latest/opp/version.info
```

输出示例：
```
bishengc++ version: 3.x.x
+------------------------------------------------------------------------------------------------+
| NPU  ID     Health       Info                                                 |
+------------------------------------------------------------------------------------------------+
| 0           OK           950                                                  |
+------------------------------------------------------------------------------------------------+
```

---

```mermaid
flowchart TB
    ROOT["/usr/local/Ascend/<br/>安装根目录"] --> TK_D["ascend-toolkit/<br/>Toolkit"]
    ROOT --> CANN["cann/<br/>环境脚本"]
    ROOT --> DRV["driver/<br/>驱动"]
    TK_D --> LATEST["latest/<br/>版本软链接"]
    LATEST --> BIN["bin/<br/>bishengc++"]
    LATEST --> INC["include/<br/>头文件"]
    LATEST --> LIB["lib64/<br/>运行时库"]
    LATEST --> OPP["opp/<br/>算子原型库"]
    CANN --> SH["set_env.sh<br/>统一入口"]
    SH --> ENV["PATH + LD_LIBRARY_PATH<br/>+ PYTHONPATH + ..."]
```

## 小结

1. **Toolkit 包**：通过 `--install --full` 安装，`--install-path` 指定路径
2. **Ops 包**：通过 `--install` 安装，路径必须与 Toolkit 一致
3. **环境变量**：`source cann/set_env.sh` 统一生效，追加到 `.bashrc` 可永久生效

---

上一节：[2.3 算子目录结构](./02.03_operator_directory_structure.ipynb) | 下一节：[2.5 章节实践](./02.05_chapter_practice.ipynb)